# stellar delta shift

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Constant stellar-observable shift

Can a constant change in stellar normalization explain part of FLAMINGO-to-TNG posterior bias? This is evaluation-only with two existing FLAMINGO M200c models: stellar sum and individual stellar masses. No model is refitted. The Delta calibration uses separate training rows; the same 231 TNG test pairs are used at every shift.

The previously measured ~0.35 dex discrepancy is a **vertical bound-mass offset**. It is not itself the appropriate horizontal stellar shift. Those old diagnostics used all TNG pairs, so we calibrate the shift afresh without final held-out targets.


In [ ]:
MODE='prepare'

In [ ]:
import os, sys, json, hashlib, pickle
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
candidates = [Path(os.environ.get('LG_PROJECT', Path.cwd())), Path.cwd(), *Path.cwd().parents]
PROJECT = next(p.resolve() for p in candidates if (p/'src/lg_npe').is_dir())
sys.path.insert(0, str(PROJECT/'src'))
from lg_npe.final_design import *
from lg_npe.registry import environment_record, write_json
from lg_npe.features import feature_names
from lg_npe.evaluation import posterior_metrics
from lg_npe.inference import posterior_draws
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Project:', PROJECT)

In [ ]:
frames = load_final_frames(PROJECT)
splits = final_splits(PROJECT, frames)
population_rows = []
for domain, frame in frames.items():
    for part in ['train','validation','test']:
        ix=splits[domain][part]; selected=frame.iloc[ix]
        population_rows.append(dict(domain=domain,split=part,n=len(ix),
            same_host=int(selected.same_host.sum()),different_host=int((~selected.same_host).sum()),
            components=len(np.unique(splits[domain]['groups'][ix])),
            target_min=selected.log_m200c.min(),target_max=selected.log_m200c.max()))
display(pd.DataFrame(population_rows))
print('Target:', TARGET_DEFINITION)

In [ ]:
BUNDLE = PROJECT/'data/derived/final_experiment_v1'
BUNDLE.mkdir(parents=True,exist_ok=True)
arrays={f'{d}_{k}':v for d,parts in splits.items() for k,v in parts.items()}
identity={d:frame_digest(f) for d,f in frames.items()}
identity['historical_split_sha256']=sha256(PROJECT/HISTORICAL_SUM/'split_indices.npz')
split_path=BUNDLE/'splits.npz'; identity_path=BUNDLE/'identity.json'
if split_path.exists() or identity_path.exists():
    if not (split_path.exists() and identity_path.exists()):
        raise RuntimeError('Incomplete frozen split bundle; inspect before rerunning')
    assert json.loads(identity_path.read_text()) == identity, 'Frozen data identities changed'
    with np.load(split_path) as saved:
        for k,v in arrays.items(): np.testing.assert_array_equal(saved[k],v)
else:
    np.savez_compressed(split_path,**arrays)
    write_json(identity_path,identity)
print('Frozen split bundle:', BUNDLE)

## Independent mapping calibration

Here 'independent' means independent of held-out targets and NPE errors, not an externally measured physical correction. Regress log combined bound-subhalo mass against separation, vr, vt, log stellar sum and host class separately in the two **training** populations. At their equal-domain reference centre, Delta is the TNG-minus-FLAMINGO intercept gap divided by the FLAMINGO stellar slope in dex/dex.

This is a constant reference-point alignment of forward regressions. It does not assume that inverting a noisy conditional regression recovers a generative stellar relation. The calibration uses all training rows; unlike the earlier diagnostic it is not restricted using a classifier trained partly on held-out features. Fit uncertainty comes from 500 component resamples; the reference stays fixed.


In [ ]:
calibration_path=BUNDLE/'delta_calibration.json'
training={d:f.iloc[splits[d]['train']].copy() for d,f in frames.items()}
# Preserve the original calibration and validate its migrated provenance separately.
validate_frozen_calibration(PROJECT, training)
calibration=json.loads(calibration_path.read_text())
delta_draws=np.load(BUNDLE/'delta_calibration_draws.npy')
DELTA=calibration['delta']
DELTAS=np.unique(np.r_[np.round(np.arange(-.10,.6001,.05),8),0.,DELTA])
assert MODE in ['prepare','evaluate']
display(pd.DataFrame([calibration])[['delta','bound_gap','flamingo_stellar_slope','interval95','n_flamingo','n_tng']])
print('Frozen grid:',DELTAS)

## Exact transformation and checkpoint provenance

Adding Delta to each log stellar mass also adds Delta to log stellar sum. It cancels exactly from log(Mstar,secondary/Mstar,primary). Apply the change **before** the historical FLAMINGO scaler; never refit that scaler or shift truth. The fixed grid was specified before posterior evaluation. No optimum is used to define the correction.

The existing NPE models trained on the larger historical FLAMINGO outer-training population. They have never fitted TNG. Their original row-validation limitations remain part of this fixed-model diagnostic, while the six new models use corrected validation.


In [ ]:
RUNS={'stellar_sum':PROJECT/HISTORICAL_SUM,'stellar_individual':PROJECT/HISTORICAL_INDIVIDUAL}
model_provenance={}
for label,run in RUNS.items():
    manifest=json.loads((run/'run_manifest.json').read_text())
    names=manifest['feature_names']
    assert names==feature_names('dynamics',label)
    assert manifest['target_column']=='log_m200c'
    with np.load(run/'split_indices.npz') as saved:
        np.testing.assert_array_equal(saved['test'],splits['flamingo']['test'])
    model_provenance[label]=dict(run=str(run.relative_to(PROJECT)),features=names,
        model_sha256=sha256(run/'model.pkl'),manifest_sha256=sha256(run/'run_manifest.json'),
        split_sha256=sha256(run/'split_indices.npz'))
    original=frames['tng'][names].iloc[:3].to_numpy(float)
    shifted=shift_stellar_features(original,names,DELTA)
    np.testing.assert_array_equal(original[:,:3],shifted[:,:3])
    print(label,names,'stellar change:',shifted[:,-1]-original[:,-1])
OUTPUT=PROJECT/'out/final_delta'/MODE/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
write_json(OUTPUT/'design_manifest.json',dict(mode=MODE,calibration=calibration,
    deltas=DELTAS,models=model_provenance,split_sha256=sha256(BUNDLE/'splits.npz'),
    n_test=len(splits['tng']['test']),draws=500,seed=261001,environment=environment_record()))
if MODE=='prepare': print('PENDING posterior evaluation in the pinned Roihu environment.')

## Available zero-shift reference on the held-out test rows

The earlier evaluation already saved 500 posterior draws for all TNG pairs. Select the same 231 test identities now to show the available zero-shift reference without loading a model. These archived draws use their recorded historical seed; the new sensitivity run will regenerate zero and nonzero shifts with its common seed. Do not combine archived zero with new shifted draws for the paired comparison.


In [ ]:
zero_rows=[]; zero_provenance={}
EVALUATION=PROJECT/'out/mass_comparison/posterior_evaluation/20260924T081006Z'
frame=frames['tng'].iloc[splits['tng']['test']]
for label in RUNS:
    path=EVALUATION/f'm200c_{label}'/'transfer_tng_posterior_draws.npz'
    with np.load(path) as saved:
        lookup=pd.Index(saved['pair_index'])
        assert lookup.is_unique
        positions=lookup.get_indexer(frame.pair_index)
        assert np.all(positions>=0)
        np.testing.assert_allclose(saved['truth'][positions],frame.log_m200c,rtol=1e-6)
        samples=saved['samples'][:,positions]; truth=saved['truth'][positions]
        same=frame.same_host.to_numpy()
        zero_provenance[label]=dict(path=str(path.relative_to(PROJECT)),sha256=sha256(path),seed=int(saved['seed']))
        for subset,mask in dict(pooled=np.ones(len(frame),bool),same_host=same,different_host=~same).items():
            zero_rows.append(dict(model=label,delta=0.,subset=subset,**posterior_metrics(samples[:,mask],truth[mask])))
zero=pd.DataFrame(zero_rows)
zero.to_csv(OUTPUT/'archived_zero_shift_reference.csv',index=False)
write_json(OUTPUT/'archived_zero_shift_provenance.json',zero_provenance)
display(zero[['model','subset','n','rmse','bias','coverage68','coverage95']])
fig,ax=plt.subplots(figsize=(7,3.5),layout='constrained')
ax.hist(delta_draws,bins=25,density=True,color='steelblue',alpha=.8)
ax.axvline(DELTA,color='k',ls='--',label=f'Delta = {DELTA:.4f} dex')
ax.set(xlabel='Training-calibrated stellar shift [dex]',ylabel='Bootstrap density',
       title='Mapping calibration uncertainty; not NPE transfer performance')
ax.legend(); fig.savefig(OUTPUT/'delta_calibration.png',dpi=220)
fig.savefig(OUTPUT/'delta_calibration.svg'); plt.show()

## Fixed-model evaluation

500 posterior draws per test row; common random seeds reduce avoidable Monte Carlo differences. Save every draw and aligned summary. Compute RMSE, bias, R2, C68, C95, median absolute standardized error beta, discrete rank-CDF distance, and 68/95% widths for pooled, same-host and different-host systems. Input-range excursion counts are descriptive support checks, not proof of extrapolation safety.


In [ ]:
metrics=[]; summaries={}; support=[]
if MODE=='evaluate':
    require_saved_model_environment()
    import torch
    ix=splits['tng']['test']; frame=frames['tng'].iloc[ix]
    truth=frame.log_m200c.to_numpy(); same=frame.same_host.to_numpy()
    for label,run in RUNS.items():
        model=load_model(run/'model.pkl'); names=model['feature_names']
        with np.load(run/'split_indices.npz') as saved:
            source_train=frames['flamingo'].iloc[saved['train']][names].to_numpy(float)
        np.testing.assert_allclose(model['scaler'].mean_,source_train.astype(np.float32).mean(0,dtype=np.float64),rtol=1e-5,atol=1e-5)
        low,high=source_train.min(0),source_train.max(0)
        for k,delta in enumerate(DELTAS):
            raw=shift_stellar_features(frame[names].to_numpy(float),names,delta)
            support.append(dict(model=label,delta=delta,n_outside_any_training_range=int(((raw<low)|(raw>high)).any(1).sum())))
            X=model['scaler'].transform(raw)
            torch.manual_seed(261001)
            samples=posterior_draws(model['posterior'],X,500,256,'cuda')
            summary=posterior_summary(samples,truth,frame.pair_index.to_numpy(),splits['tng']['groups'][ix],same)
            summaries[label,k]=summary
            summary.to_csv(OUTPUT/f'{label}_shift{k:02d}_summary.csv.gz',index=False)
            np.savez_compressed(OUTPUT/f'{label}_shift{k:02d}_draws.npz',samples=samples,pair_index=frame.pair_index)
            for subset,mask in dict(pooled=np.ones(len(frame),bool),same_host=same,different_host=~same).items():
                metrics.append(dict(model=label,delta=delta,subset=subset,
                    calibrated=bool(delta==DELTA),**posterior_metrics(samples[:,mask],truth[mask])))
    results=pd.DataFrame(metrics); results.to_csv(OUTPUT/'delta_metrics.csv',index=False)
    pd.DataFrame(support).to_csv(OUTPUT/'input_support.csv',index=False)
    display(results.loc[(results.delta==0)|results.calibrated])

## Primary paired contrast and sensitivity curves

The primary contrast is calibrated shift minus zero. The same component draws are used for both, before host stratification. These intervals condition on Delta and the fitted NPE; the separate calibration interval above measures uncertainty in the shift. The curve is sensitivity analysis, not a correction selected by test error. No causal interpretation is assigned to improvement.


In [ ]:
if MODE=='evaluate':
    intervals=[]; zero=int(np.flatnonzero(DELTAS==0)[0]); chosen=int(np.flatnonzero(DELTAS==DELTA)[0])
    for label in RUNS:
        ci,raw=paired_component_intervals(summaries[label,zero],summaries[label,chosen],repeats=2000)
        ci['model']=label; intervals.append(ci)
        raw.to_csv(OUTPUT/f'{label}_paired_bootstrap.csv.gz',index=False)
    pd.concat(intervals).to_csv(OUTPUT/'calibrated_vs_zero_intervals.csv',index=False)
    fig,axes=plt.subplots(2,4,figsize=(14,7),layout='constrained')
    for row,label in enumerate(RUNS):
        for ax,metric in zip(axes[row],['rmse','bias','coverage68','mean_width68']):
            for subset,f in results[results.model==label].groupby('subset'):
                ax.plot(f.delta,f[metric],marker='.',label=subset)
            ax.axvline(DELTA,color='k',ls='--',label='training-calibrated Delta')
            ax.axvline(0,color='gray',ls=':'); ax.set(xlabel='stellar shift [dex]',ylabel=metric,title=label)
            if metric=='coverage68':ax.axhline(.68,color='gray',ls=':')
            if metric=='bias':ax.axhline(0,color='gray',ls=':')
    axes[0,0].legend(fontsize=7)
    fig.savefig(OUTPUT/'delta_sensitivity.png',dpi=220); fig.savefig(OUTPUT/'delta_sensitivity.svg')
    plt.show()
    write_json(OUTPUT/'completion_manifest.json',dict(status='complete',models=model_provenance,
        calibration_sha256=sha256(calibration_path),deltas=DELTAS,draws=500,repeats=2000,
        n_test=len(ix),seed=261001,environment=environment_record()))

## Interpretation after execution

If the calibrated correction improves bias and coverage, a constant stellar normalization accounts for part of the transfer discrepancy. Residual bias, host dependence or poor widths indicates remaining mapping differences. An apparent match does not establish a feedback mechanism or resolve finder-dependent bound mass. With 48 different-host test pairs, uncertainty may remain wide. Preserve this executed notebook and the full timestamped output directory.
